# Session 2 · Data Warehousing and Data Pre-processing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/01-data-mining-with-weka/code/session-02-data-warehousing-and-preprocessing.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

### Example 1 · A mini star schema in SQL

In [1]:
import sqlite3

con = sqlite3.connect(":memory:")
cur = con.cursor()
cur.executescript("""
CREATE TABLE dim_date(date_key INTEGER PRIMARY KEY, month TEXT, year INTEGER);
CREATE TABLE dim_product(product_key INTEGER PRIMARY KEY, name TEXT, category TEXT);
CREATE TABLE dim_store(store_key INTEGER PRIMARY KEY, city TEXT);
CREATE TABLE fact_sales(date_key INT, product_key INT, store_key INT, quantity INT, amount REAL);

INSERT INTO dim_date VALUES (1,'Jan',2025),(2,'Feb',2025),(3,'Jan',2026);
INSERT INTO dim_product VALUES (1,'Notebook','Stationery'),(2,'Pen','Stationery'),(3,'Headphones','Electronics');
INSERT INTO dim_store VALUES (1,'Karachi'),(2,'Lahore');
INSERT INTO fact_sales VALUES
 (1,1,1,40,4000),(1,2,1,100,2000),(1,3,2,5,25000),
 (2,1,2,30,3000),(2,3,1,8,40000),(3,2,2,150,3000),(3,3,1,6,30000);
""")

print("Roll-up: total sales by year and category")
for row in cur.execute("""
    SELECT d.year, p.category, SUM(f.amount) AS total
    FROM fact_sales f
    JOIN dim_date d ON f.date_key = d.date_key
    JOIN dim_product p ON f.product_key = p.product_key
    GROUP BY d.year, p.category ORDER BY d.year, total DESC"""):
    print(row)

print("\nSlice: 2025 only, sales by city")
for row in cur.execute("""
    SELECT s.city, SUM(f.amount), SUM(f.quantity)
    FROM fact_sales f
    JOIN dim_date d ON f.date_key = d.date_key
    JOIN dim_store s ON f.store_key = s.store_key
    WHERE d.year = 2025 GROUP BY s.city"""):
    print(row)

Roll-up: total sales by year and category
(2025, 'Electronics', 65000.0)
(2025, 'Stationery', 9000.0)
(2026, 'Electronics', 30000.0)
(2026, 'Stationery', 3000.0)

Slice: 2025 only, sales by city
('Karachi', 46000.0, 148)
('Lahore', 28000.0, 35)


### Example 5 · The same pipeline in pandas

In [2]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "age": [25, 32, 41, 29, np.nan, 36, 23, 45, 27, 33],
    "salary": [45000, np.nan, 120000, 52000, 61000, 88000, 38000, 135000, 47000, 70000],
    "city": ["Karachi", "Lahore", "Karachi", None, "Islamabad", "Lahore", "Karachi", "Islamabad", "Lahore", "Karachi"],
    "experience": [1, 6, 15, 4, 7, 10, 0, 20, 3, np.nan],
    "promoted": ["no", "yes", "yes", "no", "yes", "yes", "no", "yes", "no", "no"],
})
print("Missing before:", df.isna().sum().to_dict())

num = ["age", "salary", "experience"]
df[num] = df[num].fillna(df[num].mean())            # like ReplaceMissingValues
df["city"] = df["city"].fillna(df["city"].mode()[0])
print("Missing after: ", df.isna().sum().to_dict())

norm = (df[num] - df[num].min()) / (df[num].max() - df[num].min())   # like Normalize
print("\nNormalised (first 4 rows):")
print(norm.head(4).round(3))

df["age_band"] = pd.cut(df["age"], bins=3, labels=["young", "middle", "senior"])  # like Discretize
print("\nAge bands:")
print(df[["age", "age_band"]].head(6))

encoded = pd.get_dummies(df[["city"]], prefix="city").astype(int)   # like NominalToBinary
print("\nOne-hot encoded city:")
print(encoded.head(4))

Missing before: {'age': 1, 'salary': 1, 'city': 1, 'experience': 1, 'promoted': 0}
Missing after:  {'age': 0, 'salary': 0, 'city': 0, 'experience': 0, 'promoted': 0}

Normalised (first 4 rows):
     age  salary  experience
0  0.091   0.072        0.05
1  0.409   0.360        0.30
2  0.818   0.845        0.75
3  0.273   0.144        0.20

Age bands:
         age age_band
0  25.000000    young
1  32.000000   middle
2  41.000000   senior
3  29.000000    young
4  32.333333   middle
5  36.000000   middle

One-hot encoded city:
   city_Islamabad  city_Karachi  city_Lahore
0               0             1            0
1               0             0            1
2               0             1            0
3               0             1            0


### Example 6 · Detecting outliers with the IQR rule

In [3]:
import pandas as pd

marks = pd.Series([62, 70, 68, 75, 71, 66, 69, 300, 73, 5], name="marks")
q1, q3 = marks.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, fences=({low}, {high})")
print("Outliers:", marks[(marks < low) | (marks > high)].tolist())
print("Capped:  ", marks.clip(low, high).tolist())

Q1=66.5, Q3=72.5, IQR=6.0, fences=(57.5, 81.5)
Outliers: [300, 5]
Capped:   [62.0, 70.0, 68.0, 75.0, 71.0, 66.0, 69.0, 81.5, 73.0, 57.5]
